#  From ChatAI to AIR·MS: Leveraging Large Language Models 

This notebook is a companion to the AIRMS training session II. It demonstrates:

- Connecting to AIR-MS with the **`airms-connect`** library  
- Basic SQL concepts and HANA-SQL queries for OMOP data
- Using ollama and AIR-MS

---

### Prerequisites to run this notebook
Before you can execute queries, you need:

1. A valid **Mount Sinai school network account**  
2. A **Minerva HPC account** (apply separately)  
3. **Approved access** to the AIRMS CDMDEID dataset (via SailPoint, linked to IRB #20-01288)  
4. Connection to the **Mount Sinai network** (on campus or via VPN)
5. A **Symantec VIP** two-factor authentication (more info [here](https://itsecurity.mssm.edu/vip-two-factor-setup/)

We strongly recommend launching Jupyter through **Open OnDemand** on Minerva (see the training PowerPoint for setup instructions). [OnDemand Getting Started.pptx](https://mtsinai-my.sharepoint.com/:p:/g/personal/eugeniaalessandrae_allevabonomi_mssm_edu/EbnO2s2KPHRErKG8sgUdDncBjZVH-AyyCDIVIM8GkiRndA?e=CAZdtb)


#### Connecting to AIR-MS

In [ ]:
# Install dotenv (if required)
%pip install python-dotenv

from airms_connect.connection import airms_connection

airms = airms_connection()

# On Minerva: establish the tunnel automatically.
# (Default for training; comment out if running locally)
minerva_login_node = 'li04e01' #select a login node
airms.on_minerva(minerva_login_node)

airms.connect()


# Part I: Introduction to SQL

---
## 0. Database Schema

A schema is like a namespace or folder for database objects. In AIR-MS, different schemas store different data modalities. This allows users to get access to the modalities they are allowed to see, while keeping all modalities linkable with each other across the database. 
To access a table, you must either:
- Fully quality the table name as `schema_name.table_name`
- Be inside its schema.

You can check your current schema via:
```sql
SELECT CURRENT_SCHEMA FROM DUMMY;
```

In AIR-MS, your data might be stored in schemas like `CDMPHI`, `CDMDEID`, `MSMDEID` ect. 

1. **Clauses (keywords)**  
   - Control the structure of the query.  
   - Examples: `SELECT`, `FROM`, `WHERE`, `ORDER BY`.  

2. **Functions (built-in operations)**  
   - Perform calculations on values.  
   - Examples: `AVG()`, `COUNT()`, `MAX()`.  

Clauses tell the database *what to do*, functions tell it *how to calculate*.

---

In [ ]:
query = "SELECT CURRENT_SCHEMA FROM DUMMY"
conn.sql(query).collect()

Every user has their own private schema on which they can save specific views. It has the same name as their usernames. If not specified via 
``` ptyhon 
airms.connect(schema=<base schema name here>)
```
the user schema is the standard schema used by airms.conn
One can can specify a baseline schema as follows:

In [ ]:
# We can save another ConnectionContext object with a different schema
# For example, the OMOP CDM schema de-identified
omop = airms.connect('CDMDEID')
query = "SELECT CURRENT_SCHEMA FROM DUMMY"
omop.sql(query).collect()

---

## 1. SQL Basics: Clauses vs. Functions

SQL has two main building blocks:

1. **Clauses (keywords)**  
   - Control the structure of the query.  
   - Examples: `SELECT`, `FROM`, `WHERE`, `ORDER BY`.  

2. **Functions (built-in operations)**  
   - Perform calculations on values.  
   - Examples: `AVG()`, `COUNT()`, `MAX()`.  

Clauses tell the database *what to do*, functions tell it *how to calculate*.

---

## 2. Core Clauses in HANA SQL

### `SELECT`
- Chooses which columns (or expressions) to display.  
- Always the first clause in a query.
```sql
SELECT person_id, year_of_birth
FROM cdmdeid.person;
```

In [ ]:
query = """
SELECT person_id, year_of_birth
FROM cdmdeid.person;
"""
omop.sql(query).head(10).collect()

### `FROM`
- Tells SQL which **table** (or view) the data comes from.

```sql
SELECT *
FROM cdmdeid.condition_occurrence;
```

In [ ]:
query = """
SELECT *
FROM condition_occurrence;
"""
omop.sql(query).head(10).collect()

### `WHERE`
- Filters rows **before** grouping or aggregation.  
- Only rows meeting the condition are included.

```sql
SELECT *
FROM cdmdeid.person
WHERE year_of_birth < 1980;
```

In [ ]:
query = """
SELECT * 
FROM cdmdeid.person
WHERE year_of_birth < 1980;
"""
omop.sql(query).head(10).collect()

### `GROUP BY`
- Groups rows together for aggregation.  
- Without `GROUP BY`, functions like `COUNT` or `AVG` summarize *all rows*.  
- With `GROUP BY`, they summarize *per group*.

```sql
SELECT gender_concept_id, COUNT(*) AS n_patients
FROM cdmdeid.person
GROUP BY gender_concept_id;
```

In [ ]:
query = """
SELECT gender_concept_id, COUNT(*) AS n_patients
FROM cdmdeid.person
GROUP BY gender_concept_id
"""
omop.sql(query).collect()

### `HAVING`
- Filters groups (not rows).  
- Used after `GROUP BY` + aggregation.

```sql
SELECT year_of_birth, COUNT(*) AS n_patients
FROM cdmdeid.person
GROUP BY year_of_birth
HAVING COUNT(*) > 1000;
```

In [ ]:
query = """
SELECT year_of_birth, COUNT(*) AS n_patients
FROM cdmdeid.person
GROUP BY year_of_birth
HAVING COUNT(*) > 1000;
"""
omop.sql(query).collect()

### `ORDER BY`
- Sorts the results by one or more columns.  
- Default = ascending (`ASC`), descending = `DESC`.

```sql
SELECT person_id, year_of_birth
FROM cdmdeid.person
ORDER BY year_of_birth DESC;
```


In [ ]:
query = """
SELECT person_id, year_of_birth
FROM cdmdeid.person
ORDER BY year_of_birth DESC;
"""
omop.sql(query).head(10).collect()

### `LIMIT` / `TOP`
- Restrict number of results.  
- In **HANA**, you can use either:

```sql
-- ANSI style
SELECT person_id, year_of_birth
FROM cdmdeid.person
ORDER BY year_of_birth
LIMIT 10;

-- HANA style
SELECT TOP 10 person_id, year_of_birth
FROM cdmdeid.person
ORDER BY year_of_birth;
```

In [ ]:
query = """
-- ANSI style
SELECT person_id, year_of_birth
FROM cdmdeid.person
WHERE birth_datetime IS NOT NULL
ORDER BY year_of_birth
LIMIT 10;
"""
omop.sql(query).collect()

In [ ]:
query = """
-- HANA style
SELECT TOP 100 person_id, birth_datetime
FROM cdmdeid.person
WHERE birth_datetime IS NOT NULL
ORDER BY birth_datetime DESC;
"""
omop.sql(query).collect()

## 3. Common SQL Functions in HANA

### `COUNT()`
- Counts rows.  
- `COUNT(*)` counts all rows.  
- `COUNT(column)` counts only non-null values.

```sql
SELECT COUNT(*) AS n_patients
FROM cdmdeid.person;
```

In [ ]:
query = """
SELECT COUNT(*) AS n_patients
FROM cdmdeid.person;
"""
omop.sql(query).collect()

### `AVG()`
- Computes the average of a numeric column.  
- Ignores `NULL` values.

```sql
SELECT AVG(value_as_number) AS avg_hemoglobin
FROM cdmdeid.measurement
WHERE measurement_concept_id = 3000963;
```

To ensure we have only numeric values we can **CAST** the datatype using
```sql
CAST(VALUE_AS_NUMBER AS DOUBLE)
```

In [ ]:
query = """
SELECT AVG(CAST(value_as_number AS DOUBLE)) AS avg_hemoglobin
FROM cdmdeid.measurement
WHERE measurement_concept_id = 3000963;
"""
omop.sql(query).collect()

### `SUM()`
- Adds values across rows.

```sql
SELECT SUM(value_as_number) AS total_lab_values
FROM cdmdeid.measurement
WHERE measurement_concept_id = 3027018; 
```

3027018 is code for Glucose

In [ ]:
query = """
SELECT SUM(CAST(value_as_number AS DOUBLE)) AS total_lab_values
FROM cdmdeid.measurement
WHERE measurement_concept_id = 3027018
"""
omop.sql(query).collect()

### `MIN()` / `MAX()`
- Get smallest / largest value.

```sql
SELECT MIN(year_of_birth) AS oldest, MAX(year_of_birth) AS youngest
FROM cdmdeid.person;
```


In [ ]:
query = """
SELECT MIN(year_of_birth) AS oldest, MAX(year_of_birth) AS youngest
FROM cdmdeid.person;
"""
omop.sql(query).collect()

### String functions
- `CONCAT(a, b)` → join two strings.  
- `UPPER(string)` → convert to uppercase.  
- `SUBSTRING(string, start, length)` → extract part of a string.

```sql
SELECT CONCAT('Patient_', person_id) AS patient_label
FROM cdmdeid.person
LIMIT 5;
```

In [ ]:
query = """
SELECT CONCAT('Nice_Little_Prefix_Here', person_id) AS patient_label
FROM cdmdeid.person
LIMIT 5;
"""
omop.sql(query).collect()

### Date functions
- `CURRENT_DATE` → today’s date.  
- `ADD_DAYS(date, n)` → shift date forward/backward.  
- `EXTRACT(YEAR FROM date)` → get part of a date.

```sql
SELECT person_id,
       ADD_DAYS(visit_start_date, 30) AS followup_date
FROM cdmdeid.visit_occurrence;
```

In [ ]:
query = """
SELECT TOP 10 person_id, ADD_DAYS(visit_start_date, 30) AS followup_date
FROM cdmdeid.visit_occurrence;
"""
omop.sql(query).collect()

---

## 4. Joins in SQL

### What is a Join?
- A **join** combines rows from two (or more) tables based on a related column.  
- In OMOP, this is common: e.g., joining **visits** with **conditions**, or **drugs** with **patients**.  
- The related column is usually an **ID** (like `person_id` or `visit_occurrence_id`).

---

### Types of Joins (in HANA)

- **INNER JOIN** → only rows that match in both tables.  
- **LEFT OUTER JOIN** → all rows from the left table, plus matches from the right (if any).  
- **RIGHT OUTER JOIN** → all rows from the right table, plus matches from the left.  
- **FULL OUTER JOIN** → all rows from both tables, matched where possible.  

In practice, **INNER JOIN** and **LEFT JOIN** are the most common in OMOP queries.

---

In [ ]:
### Example 1: Diagnoses with Visit Dates
query = """
SELECT TOP 10 c.person_id, c.condition_concept_id, v.visit_start_date
FROM cdmdeid.condition_occurrence c
INNER JOIN cdmdeid.visit_occurrence v
  ON c.visit_occurrence_id = v.visit_occurrence_id
WHERE v.visit_start_date >= '2020-01-01';
"""
omop.sql(query).collect()

In [ ]:
### Example 2: All Visits, Even Without a Condition (LEFT JOIN)
query = """
SELECT TOP 10 v.person_id, v.visit_start_date, c.condition_concept_id
FROM cdmdeid.visit_occurrence v
LEFT JOIN cdmdeid.condition_occurrence c
  ON v.visit_occurrence_id = c.visit_occurrence_id
"""

omop.sql(query).collect()

In [ ]:
### Example 3: Patients with Metformin and Diabetes
query = """
SELECT TOP 10 DISTINCT d.person_id
FROM cdmdeid.drug_exposure d
JOIN cdmdeid.condition_occurrence c
  ON d.person_id = c.person_id
WHERE d.drug_concept_id = 40164897   
  AND c.condition_concept_id = 201826; 
"""
omop.sql(query).collect()

In [ ]:
### Example 4: Nested Join — Lab Results During Visits
query = """
SELECT TOP 10 m.person_id,
       m.value_as_number AS hemoglobin,
       v.visit_start_date
FROM cdmdeid.measurement m
JOIN cdmdeid.visit_occurrence v
  ON m.visit_occurrence_id = v.visit_occurrence_id
WHERE m.measurement_concept_id = 3000963;
"""
omop.sql(query).collect()

---

## 5. Creating New Columns (Derived Columns)

SQL lets you **create new columns on the fly** by applying functions or expressions in the `SELECT` clause.  
These are sometimes called **computed/derived columns** or **virtual columns**.


### General Syntax
```sql
SELECT column1,
       column2,
       expression AS new_column_name
FROM table_name;
```

- `expression` can be:
  - A **function** (`AVG()`, `CONCAT()`, `DAYS_BETWEEN()`).  
  - An **arithmetic operation** (`+`, `-`, `*`, `/`).  
  - A **case expression** (`CASE WHEN ... THEN ... END`).


In [ ]:
#### 1. Date Differences
query = """
SELECT person_id,
       visit_start_date,
       visit_end_date,
       DAYS_BETWEEN(visit_start_date, visit_end_date) AS visit_length_days
FROM cdmdeid.visit_occurrence
LIMIT 5;
"""
omop.sql(query).collect()

In [ ]:
#### 2. Arithmetic on Numeric Columns with 3027018 (glucose)
query = """
SELECT measurement_id,
       value_as_number,
       unit_source_value,
       value_as_number * 0.18 AS value_in_mmolL
FROM cdmdeid.measurement
WHERE measurement_concept_id = 3000483
AND unit_source_value='mg/dL'
LIMIT 100;
"""
omop.sql(query).collect()

In [ ]:
#### 3. Conditional New Column
query = """
SELECT person_id,
       year_of_birth,
       CASE 
         WHEN year_of_birth < 1970 THEN 'Older'
         ELSE 'Younger'
       END AS age_group
FROM cdmdeid.person
LIMIT 10;
"""
omop.sql(query).collect()

In [ ]:
#### 4. Combining Multiple Columns from Multiple Tables
query = """
SELECT m.person_id,
       m.value_as_number,
       v.visit_start_date,
       DAYS_BETWEEN(p.birth_datetime, v.visit_start_date) AS age_in_days
FROM cdmdeid.measurement m
JOIN cdmdeid.person p 
    ON m.person_id = p.person_id
JOIN cdmdeid.visit_occurrence v 
    ON m.visit_occurrence_id = v.visit_occurrence_id
LIMIT 5;
"""
omop.sql(query).collect()

# Part II: LLMs and AIR-MS
This section showcases how to use ollama and AIR-MS within a single python session in Minerva. 
Ollama is a server of open-source models that one can run locally on Minerva. 
This allows researchers to use the power of LLMs in their research workflows, while maintaining privacy through local hosting on Minerva. 

## Setting Up OLLAMA on Minerva
To setup ollama on minerva, first navigate to `Clusters>Chimera Shell Access` on OnDemand. You will be prompted to enter your passowrd. This consists of your Mount Sinai password **AND** the **SYMANTEC VIP token**. If you don't have a Symanted VIP two-factor authentication setup, please follow [these](https://itsecurity.mssm.edu/vip-two-factor-setup/) instructions. 

Once you are logged into Minerva via the terminal, you will need to run the following command:
```bash
minerva-ollama-web.sh -o <your ollama preffered location here>
```

The location defined after -o should be somewhere where you have enough free space to download different LLMs. If you didn't saturate your `work` folder yet, a good starting point can be that folder (`/sc/arion/work/<your user name>`)

After running the above command, you will see something like:
```
$ sh minerva-ollama-web.sh 
[INFO] Image not specified, check if previously used 
[INFO] Found previously used image /hpc/users/hasans10/minerva_jobs/ollama_jobs/ollama_v0.3.10.sif. Using it. 
[INFO] Project is not specified, or is acc_null, using 1st avail project. 
[INFO] Project to use is acc_hpcstaff 
[INFO] Parameters used are: 
[INFO] -n       4
[INFO] -M       300 
[INFO] -W       6:00
[INFO] -P       acc_hpcstaff 
[INFO] -J       ollama 
[INFO] -q       gpu 
[INFO] -R       v100 
[INFO] -g       1 
[INFO] -o       /hpc/users/hasans10/minerva_jobs/ollama_jobs
[INFO] -i       
/hpc/users/hasans10/minerva_jobs/ollama_jobs/ollama_v0.3.10.sif 
[INFO] Submitting Ollama job... 
[INFO] Job ID: 189591101 
Ollama is started on compute node lg03a03, port 9999 
[Authentication Info] 
User ID: hasans10 
Token: d6972ea4292e66e93d4ece5f15831f14 
Access URL: http://10.95.46.94:51101 

*** You can access Ollama from Python as shown below ***** 
from ollama import Client
ollama_client = Client(host='http://10.95.46.94:51101', headers={"Authorization": "Bearer hasans10:d6972ea4292e66e93d4ece5f15831f1"})
```

### you will need to copy the following line
```bash
ollama_client = Client(host='http://10.95.46.94:51101', headers={"Authorization": "Bearer hasans10:d6972ea4292e66e93d4ece5f15831f1"})
```
and past it in the next cell

In [ ]:
# Setting Up OLLAMA on Minerva
from ollama import Client
### PASTE THE COPIED LINES FROM THE TERMINAL HERE
ollama_client = Client(host='http://10.95.46.94:52602', headers={"Authorization": "Bearer alleve01:53a12ec586630ef5ce66e3a77630dc14"})

To use a open source model you will first need to pull it and load it. You can find a list of available models [here](https://ollama.com/search)
Let's start by trying  `mistral` out. 

In [ ]:
# download the model
model = 'mistral'
ollama_client.pull(model)

Let's first build a python class to make the interaction with the model easier. We will call is 'Chat', and this class will take care of extracting the models' response and translating it into a nice structured text

In [ ]:
# Chat Python Class to make interaction easier
from IPython.display import Markdown, display
from dataclasses import dataclass, field

@dataclass
class Chat:
    model: str = "mistral"
    system: str = "You are a concise, helpful assistant."
    history: list = field(default_factory=list)

    def ask(self, msg, stream=False, **opts):
        messages = [{"role":"system","content":self.system}, *self.history, {"role":"user","content":msg}]
        if stream:
            acc = []
            for part in ollama_client.chat(model=self.model, messages=messages, stream=True, options=opts, keep_alive=0):
                chunk = part.get("message", {}).get("content", "")
                acc.append(chunk)
                print(chunk, end="")   # live streaming
            print()
            reply = "".join(acc)
            self.history += [{"role":"user","content":msg}, {"role":"assistant","content":reply}]
            return reply
        else:
            resp = ollama_client.chat(model=self.model, messages=messages, options=opts)
            reply = resp["message"]["content"]
            self.history += [{"role":"user","content":msg}, {"role":"assistant","content":reply}]
            return display(Markdown(reply))

We can now instantiate a chat object for mistral and interact with it via the `.ask` method

In [ ]:
mistral = Chat(model='mistral', system="You are a concise, helpful assistant.")
# here system can be customized to set the behavior of the model

In [ ]:
mistral.ask("What is OMOP common data model?")

Because we are working in the same environment with both ollama and AIR-MS, we can let those tools interact with each other. For example, we might want to ask Mistral to help us build an SQL query to extract all female-born patients, born between 1990 and 2000

In [ ]:
mistral.ask("""Can you help me write an SQL query to extract all female patients, born between 1990 and 2000? This information
            is stored in the 'person' table of the CDMDEID schema, where females are identified by gender_concept_id = 8532 and 
            year_of_birth is the year of birth.
            """)

In [ ]:
#let's try something more complex
mistral.ask("""Can you help me write an SQL query to extract all female patients, born between 1990 and 2000? This information
            is stored in the 'person' table of the CDMDEID schema, where females are identified by gender_concept_id = 8532 and 
            year_of_birth is the year of birth. They should be also affected by diabetes mellitus type I
            """)

### Specialized LLMs for text-to-SQL
There are several specialized LLM's that have been trained to produce SQL queries. For example `a-kore/Arctic-Text2SQL-R1-7B:latest`. Let's see if this model is better at building this query.

In [ ]:
model = 'a-kore/Arctic-Text2SQL-R1-7B:latest'
ollama_client.pull(model) 
artic = Chat(model=model, system="You are a helpful assistant that translates natural language to SQL queries.")

In [ ]:
artic.ask("""Can you help me write an SQL query to extract all female patients, born between 1990 and 2000? This information
            is stored in the 'person' table of the CDMDEID schema, where females are identified by gender_concept_id = 8532 and 
            year_of_birth is the year of birth. They should be also affected by diabetes mellitus type I
            """)

### Improved prompting
SQL comes in many dialects, and OMOP is a standardized data model. We can help our models to build meaningful SQL queries to answer our questions by giving them the right context. We have prepared a little text file with one of such prompts, which includes information about OMOP, AIRMS and HANA SQL. 

To do this, download the prompt text [here](https://mtsinai-my.sharepoint.com/:t:/g/personal/eugeniaalessandrae_allevabonomi_mssm_edu/EVdVAdemvs1KvFIgU5Gs-g4BDh-rf_U6fYMbgNFyKLp3fg?e=MzruGy) and load it into your working folder.

Let's see if this improves our model's output.

In [ ]:
# read context text
with open("omop_context.txt", "r", encoding="utf-8") as f:
    context = f.read()

In [ ]:
artic_omop = Chat(model=model, system=context)

In [ ]:
artic_omop.ask("""Can you help me write an SQL query to extract all female patients and age as a number, born between 1990 and 2000? This information
            is stored in the 'person' table of the CDMDEID schema, where females are identified by gender_concept_id = 8532 and 
            year_of_birth is the year of birth. They should be also affected by diabetes mellitus type I
            """)

In [ ]:
# What we actually want is
query = """
SELECT DISTINCT p.person_id,
                2023 - p.year_of_birth AS age
FROM CDMDEID.PERSON p
INNER JOIN CDMDEID.CONDITION_OCCURRENCE c ON p.person_id = c.person_id
WHERE p.gender_concept_id = 8532 
AND p.year_of_birth BETWEEN 1990 AND 2000
AND c.condition_concept_id = 201826
"""
omop.sql(query).head(10).collect()

In [ ]:
# let's store this as a dataframe that can be used for analysis
df = omop.sql(query).head(1000).collect() #we are just going to collect the first 1000 patients

### More then SQL - Vibe Coding with ollama

Using ollama-hosted LLMs to help with SQL is just one of the many things one can do. 
For example, ollama can help you **vibe code**.
Vibe coding is an informal, intuitive approach to programming where you rely on the “feel” or flow of the problem rather than a strict plan or detailed specification. It emphasizes creativity, experimentation, and rapid iteration over rigid structure or best practices.With LLMs, vibe coding means writing code by “feeling out” what works through natural-language prompts instead of carefully planning everything in advance. You describe what you want, let the model generate code, then iteratively adjust based on the model’s output and your intuition.

In [ ]:
wizard_model = "jcdickinson/wizardcoder:latest"
ollama_client.pull(wizard_model)

In [ ]:
wizard = Chat(model=wizard_model, system="You are a coding assistant providing me with useful python code to solve my task")

In [ ]:
wizard.ask("""
Please write the code for a nice plot for age distribution. 
The column is called AGE and is stored in a dataframe object called df'
""")

In [ ]:
# try the output out

### Chat vibes in jupyter notebooks
You can also interact with LLMs more consistently by using this **chatbot** function. 

In [ ]:
def chatbot(chat: Chat, **opts):
    import ipywidgets as w
    from IPython.display import display
    import html

    box = w.VBox(layout=w.Layout(border='1px solid #ddd', height='420px', overflow_y='auto'))
    inp = w.Text(placeholder="Ask me anything…")
    btn = w.Button(description="Send")

    def send(user_text: str):
        if not user_text or not user_text.strip():
            return

        user_html = w.HTML(f"<b>You:</b> {html.escape(user_text)}")
        box.children += (user_html,)

        bot_widget = w.HTML("<b>Bot:</b> ")
        box.children += (bot_widget,)

        acc = []
        try:
            messages = [{"role": "system", "content": chat.system},
                        *chat.history,
                        {"role": "user", "content": user_text}]

            for part in ollama_client.chat(
                model=chat.model,
                messages=messages,
                stream=True,
                options=opts,
                keep_alive=0
            ):
                chunk = part.get("message", {}).get("content", "")
                if not chunk:
                    continue
                acc.append(chunk)
                bot_widget.value = "<b>Bot:</b> " + html.escape("".join(acc)).replace("\n", "<br>")

            reply = "".join(acc)
            chat.history += [
                {"role": "user", "content": user_text},
                {"role": "assistant", "content": reply},
            ]

        except Exception as e:
            bot_widget.value = f"<b>Bot:</b> <i>Error:</i> {html.escape(str(e))}"

    #  multiline 
    inp = w.Textarea(placeholder="Ask me anything…", layout=w.Layout(height='60px', width='1000px'))
    def on_textarea_change(change):
        if change["name"] == "value" and change["new"].endswith("\n"):
            text = inp.value.strip()
            inp.value = ""
            send(text)
    inp.observe(on_textarea_change)

    def on_click(_):
        text = inp.value
        inp.value = ""
        send(text)

    def on_enter(change):
        # Called whenever value changes. Only trigger if Enter was pressed (value non-empty, and we just committed).
        if change["type"] == "change" and change["name"] == "value":
            text = change["new"].strip()
            if text:
                inp.value = ""
                send(text)

    btn.on_click(on_click)
    inp.continuous_update = False
    inp.observe(on_enter, names="value")

    display(w.VBox([inp, btn, box]))

In [ ]:
chatbot(wizard)